In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

In [20]:
DATA_PATH = "./notebooks/book-text-ml/book_bestseller_clean.csv"

df_books = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print("데이터 크기:", df_books.shape)

print("컬럼:", df_books.columns.tolist())

df_books[["상품명", "분야"]].head(10)

데이터 크기: (199, 8)
컬럼: ['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']


,상품명,분야
0,소년이 온다,소설
1,모순,소설
2,결국 국민이 합니다,정치/사회
3,혼모노,소설
4,급류,소설
5,초역 부처의 말,인문
6,청춘의 독서(특별증보판),인문
7,어른의 행복은 조용하다,시/에세이
8,채식주의자,소설
9,단 한 번의 삶(강물에디션 활판인쇄 한정판),시/에세이


In [21]:
df_model = df_books[["상품명", "분야"]].copy()

for col in ["상품명", "분야"]:

    df_model[col] = (

        df_model[col]

        .fillna("")

        .astype(str)

        .str.strip()

    )

df_model = df_model[

    (df_model["상품명"] != "") &

    (df_model["분야"] != "")

].reset_index(drop=True)

print("모델링 데이터 크기:", df_model.shape)

모델링 데이터 크기: (199, 2)


In [22]:

print("분야 종류 수:", df_model["분야"].nunique())

df_model["분야"].value_counts().head(20)

분야 종류 수: 16


분야
소설         48
인문         27
경제/경영      26
시/에세이      25
자기계발       17
외국어        15
어린이(초등)    10
정치/사회       7
청소년         5
과학          5
역사/문화       5
가정/육아       3
요리          2
만화          2
예술/대중문화     1
컴퓨터/IT      1
Name: count, dtype: int64

In [23]:
class_counts = df_model["분야"].value_counts()

class_counts[class_counts < 2]

분야
예술/대중문화    1
컴퓨터/IT     1
Name: count, dtype: int64

In [26]:
X = df_model["상품명"]
y = df_model["분야"]

print(df_model)

                                상품명     분야
0                            소년이 온다     소설
1                                모순     소설
2                        결국 국민이 합니다  정치/사회
3                               혼모노     소설
4                                급류     소설
..                              ...    ...
194                       용의자 X의 헌신     소설
195  당신은 결국 무엇이든 해내는 사람(특별 리커버 에디션)  시/에세이
196                        정의란 무엇인가     인문
197            당신에게 분명 좋은 일만 생길 거예요  시/에세이
198              결핍은 우리를 어떻게 변화시키는가  경제/경영

[199 rows x 2 columns]


In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (159,)
Test : (40,)


In [32]:
tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)

X_test_tfidf = tfidf.transform(X_test)

print("Train TF-IDF:", X_train_tfidf.shape)

print("Test TF-IDF :", X_test_tfidf.shape)

print(type(X_train_tfidf))

Train TF-IDF: (159, 435)
Test TF-IDF : (40, 435)
<class 'scipy.sparse._csr.csr_matrix'>


In [33]:
model = MultinomialNB()

In [40]:
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)

In [35]:
result = pd.DataFrame({
    "상품명": X_test.reset_index(drop=True),
    "실제_분야": y_test.reset_index(drop=True),
    "예측_분야": y_pred,
})

result.head(20)

,상품명,실제_분야,예측_분야
0,나는 미국 월배당 ETF로 40대에 은퇴한다,경제/경영,소설
1,위버멘쉬,인문,소설
2,부자 아빠 가난한 아빠(20주년 특별 기념판),경제/경영,소설
3,천 개의 파랑,소설,소설
4,인간 실격,소설,소설
5,월가의 영웅,경제/경영,소설
6,긴긴밤,어린이(초등),소설
7,영어로 문장 만들기 훈련 1차 임계점,외국어,소설
8,이어령의 말,인문,소설
9,세상은 실제로 어떻게 돌아가는가,인문,소설


In [43]:
y_test.value_counts()

분야
경제/경영      9
소설         8
인문         4
자기계발       4
어린이(초등)    3
시/에세이      3
과학         2
정치/사회      2
외국어        1
역사/문화      1
청소년        1
가정/육아      1
만화         1
Name: count, dtype: int64

In [41]:
# y_pred.value_counts()
pd.Series(y_pred).value_counts()

소설    40
Name: count, dtype: int64

In [44]:

report = classification_report(

    y_test,

    y_pred,

    zero_division=0,

)

print(report)

              precision    recall  f1-score   support

       가정/육아       0.00      0.00      0.00         1
       경제/경영       0.00      0.00      0.00         9
          과학       0.00      0.00      0.00         2
          만화       0.00      0.00      0.00         1
          소설       0.20      1.00      0.33         8
       시/에세이       0.00      0.00      0.00         3
     어린이(초등)       0.00      0.00      0.00         3
       역사/문화       0.00      0.00      0.00         1
         외국어       0.00      0.00      0.00         1
          인문       0.00      0.00      0.00         4
        자기계발       0.00      0.00      0.00         4
       정치/사회       0.00      0.00      0.00         2
         청소년       0.00      0.00      0.00         1

    accuracy                           0.20        40
   macro avg       0.02      0.08      0.03        40
weighted avg       0.04      0.20      0.07        40



In [42]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.2000


In [45]:
labels = model.classes_

cm = confusion_matrix(

    y_test,

    y_pred,

    labels=labels,

)

df_cm = pd.DataFrame(

    cm,

    index=labels,

    columns=labels,

)

df_cm

,가정/육아,경제/경영,과학,만화,소설,시/에세이,어린이(초등),역사/문화,예술/대중문화,외국어,요리,인문,자기계발,정치/사회,청소년,컴퓨터/IT
가정/육아,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
경제/경영,0,0,0,0,9,0,0,0,0,0,0,0,0,0,0,0
과학,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0
만화,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
소설,0,0,0,0,8,0,0,0,0,0,0,0,0,0,0,0
시/에세이,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0
어린이(초등),0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0
역사/문화,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
예술/대중문화,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
외국어,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0


In [46]:

misclassified = result[

    result["실제_분야"] != result["예측_분야"]

].copy()

print("오분류 수:", len(misclassified))

misclassified.head(20)

오분류 수: 32


,상품명,실제_분야,예측_분야
0,나는 미국 월배당 ETF로 40대에 은퇴한다,경제/경영,소설
1,위버멘쉬,인문,소설
2,부자 아빠 가난한 아빠(20주년 특별 기념판),경제/경영,소설
5,월가의 영웅,경제/경영,소설
6,긴긴밤,어린이(초등),소설
7,영어로 문장 만들기 훈련 1차 임계점,외국어,소설
8,이어령의 말,인문,소설
9,세상은 실제로 어떻게 돌아가는가,인문,소설
10,책문,역사/문화,소설
11,에그박사 14,어린이(초등),소설


### 실습 16. 예측 결과 저장하기

In [47]:

result.to_csv(

    "chapter04_predictions.csv",

    index=False,

    encoding="utf-8-sig",

)

misclassified.to_csv(

    "chapter04_misclassified.csv",

    index=False,

    encoding="utf-8-sig",

)

In [48]:
new_titles = [

    "파이썬으로 시작하는 데이터 분석",

    "처음 배우는 주식 투자",

    "마음을 이해하는 심리학",

]

new_vectors = tfidf.transform(new_titles)

new_predictions = model.predict(new_vectors)

pd.DataFrame({

    "상품명": new_titles,

    "예상_분야": new_predictions,

})

,상품명,예상_분야
0,파이썬으로 시작하는 데이터 분석,소설
1,처음 배우는 주식 투자,소설
2,마음을 이해하는 심리학,소설
